# [M6] DÙNG LLM NHƯ TRỢ LÝ HỌC VÀ THI
## Buổi S02: Quy Trình Debug Code Thi Đấu Bằng LLM & Đối Soát Ảo Giác

---

### Metadata Khóa Học
* **Module:** M6 — DÙNG LLM NHƯ TRỢ LÝ HỌC VÀ THI
* **Buổi:** S02
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook quy trình Debugging bằng LLM và kiểm thử biên an toàn
* **Mức độ liên quan VOAI / Olympic AI:** Rút ngắn thời gian xử lý sự cố trong phòng thi từ hàng giờ xuống dưới 1 phút.

#### Mục tiêu học tập (Learning Objectives)
- Làm chủ quy trình 4 bước chuẩn mực khi yêu cầu LLM hỗ trợ gỡ lỗi: Báo cáo vết lỗi (Traceback) + Mã nguồn tối thiểu (MRE) + Bối cảnh môi trường + Giả thuyết ban đầu.
- Sử dụng LLM để tự động sinh các Bộ dữ liệu kiểm thử biên (Edge Cases & Synthetic Test Cases) nhằm phát hiện lỗi tiềm ẩn.
- Thiết lập quy trình 3 bước Đối soát Ảo giác (Hallucination Cross-Verification Protocol).
- Tự tay cô lập và sửa chữa các lỗi phức tạp trong pipeline PyTorch bằng quy trình gỡ lỗi có trợ lực AI.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M6_S01 (Khung RTCC Prompt).
- Hiểu cách đọc thông báo lỗi (Error Traceback) trong Python.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Khi gặp lỗi code trong phòng thi, phản xạ thông thường của một thí sinh hoảng loạn là: Chụp một bức ảnh màn hình mờ căm hoặc ném nguyên một file code 500 dòng vào ChatGPT và gõ: *"Code bị lỗi, sửa đi!"*. Kết quả bạn nhận lại sẽ là một mớ hỗn độn bịa đặt, cắt xén mất các hàm quan trọng, và bạn tốn thêm 1 tiếng đồng hồ nữa trong tuyệt vọng.

Gỡ lỗi (Debugging) là một môn khoa học chính xác. Một kỹ sư chuyên nghiệp biết cách **đóng gói lỗi thành một Mẫu tái lập tối thiểu (Minimal Reproducible Example - MRE)** và dẫn dắt LLM khoanh vùng đúng nguyên nhân gốc rễ trong chưa đầy 30 giây. Làm chủ quy trình Debugging bằng LLM giúp bạn biến sự cố thành cơ hội và tiết kiệm hàng giờ quý giá trong phòng thi.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Cấu trúc Gói Báo cáo Lỗi Chuẩn (Structured Bug Report Packet)
Khi gửi lỗi cho LLM, gói thông tin bắt buộc phải gồm 4 phần:
1. **Full Traceback:** Toàn bộ thông báo lỗi từ dòng đầu đến dòng cuối (chứa tên file, dòng bị lỗi và loại ngoại lệ `Exception`).
2. **Minimal Reproducible Example (MRE):** Đoạn mã ngắn nhất có thể tự chạy độc lập để tái lập đúng lỗi đó (loại bỏ toàn bộ các phần code râu ria không liên quan).
3. **Environment & Tensor Shapes:** Phiên bản thư viện (PyTorch 2.0, Python 3.10) và kích thước thực tế của các biến đầu vào ngay trước dòng bị lỗi.
4. **My Hypothesis (Giả thuyết cá nhân):** *"Tôi nghi ngờ lỗi này do chiều kênh màu của Tensor bị đảo lộn ở bước Permute"*.

### 2. Sinh Test Cases nhân tạo (Synthetic Edge-Case Generation)
Hầu hết các lỗi sập code trong phòng thi xảy ra do các trường hợp biên dị thường:
* Batch chỉ có đúng 1 mẫu ($N=1$ làm sập BatchNorm).
* Câu văn bản rỗng (Độ dài bằng 0).
* Ảnh có toàn bộ pixel bằng nhau (Độ tương phản bằng 0).
* Dữ liệu chứa toàn giá trị âm hoặc giá trị cực lớn ($10^9$).
*Yêu cầu LLM sinh ra các bộ test biên này giúp bạn kiểm thử độ bền (Robustness) của pipeline trước khi nộp bài!*



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Quy trình LLM Debugging Workflow khép kín:
[Code bị Lỗi Runtime / Crash]
             │
             ▼
[Thu nhỏ thành MRE (Mẫu Tái lập Tối thiểu)] ──> [Đóng gói: Traceback + Shape + Environment]
                                                                │
                                                                ▼
                                                    [Gửi gói Debug cho LLM]
                                                                │
                                                                ▼
[Đối soát Ảo giác (Docstring / Sandbox Run)] ◄────── [LLM đề xuất Giải pháp]
             │
             ▼
[Kiểm nghiệm trên Synthetic Edge Cases] ──> [Áp dụng an toàn vào Bài thi chính!]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
# 1. MO PHONG TINH HUONG DEBUG THUC TE: LOI LECH CHIEU TRONG PYTORCH
# Doan code gay loi duoc dong goi thanh Minimal Reproducible Example (MRE)

import torch
import torch.nn as nn

# Tinh huong loi: Kich thuoc dau vao khong khop voi kien truc mang
def reproduce_buggy_code():
    try:
        model = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
        # Dau vao bi sai chieu: Chi co 8 dac trung thay vi 10
        wrong_input = torch.randn(4, 8) 
        output = model(wrong_input)
    except Exception as e:
        import traceback
        error_trace = traceback.format_exc()
        return error_trace

traceback_output = reproduce_buggy_code()
print("===== THONG BAO LOI THUC TE (TRACEBACK THU THAP DUOC) =====")
print(traceback_output)

### 🔍 Code Walkthrough (Khối 1)
Khối code mô phỏng việc trích xuất vết lỗi (Traceback) thực tế một cách chuẩn xác:
- Sử dụng module `traceback` để thu thập đầy đủ thông tin lỗi hệ thống.
- Cung cấp dữ liệu thô trung thực sẵn sàng nạp vào khung gỡ lỗi có cấu trúc.


In [ ]:
# 2. FRAMEWORK TAO GOI DEBUG CHUAN MUC GUI CHO LLM (STRUCTURED BUG PACKET)
def create_debug_prompt(code_snippet, error_trace, env_info, hypothesis):
    prompt = f"""### SYSTEM CONTEXT:
Ban la Chuyen gia Debugging He thong AI cap cao. Nhiem vu cua ban la phan tich nguyen nhan goc re cua loi sau va de xuat huong sua chua an toan nhat.

### 1. MINIMAL REPRODUCIBLE EXAMPLE (MRE):
```python
{code_snippet}
```

### 2. FULL ERROR TRACEBACK:
```
{error_trace}
```

### 3. ENVIRONMENT & RUNTIME METADATA:
- Thiet bi & Phien ban: {env_info}

### 4. MY HYPOTHESIS & OBSERVATION:
{hypothesis}

### 5. DEBUGGING INSTRUCTIONS FOR LLM:
- Hay chi ra dong code cu the gay ra loi va ban chat toan hoc ben duoi.
- Giai thich tai sao ma tran dau vao lai khong the nhan voi ma tran trong so.
- De xuat dung 2 dong code sua loi ma khong can viet lai toan bo chuong trinh.
- Cung cap 1 doan code kiem thu bien (Edge-case test) de phong ngua loi nay lap lai.
"""
    return prompt

bug_code = '''model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
wrong_input = torch.randn(4, 8)
output = model(wrong_input)'''

debug_prompt = create_debug_prompt(
    code_snippet=bug_code,
    error_trace=traceback_output,
    env_info="PyTorch 2.1.0, CPU, OS Windows",
    hypothesis="Toi thay kich thuoc dac trung dau vao (8) khong khop voi in_features (10) cua tang Linear dau tien."
)

print("===== GOI PROMPT DEBUG HOAN HAO SAN SANG GUI CHO LLM =====")
print(debug_prompt)

### 🔍 Code Walkthrough (Khối 2)
Khối code tự động đóng gói toàn bộ thông tin gỡ lỗi thành một tài liệu kỹ thuật hoàn chỉnh:
- Giúp LLM nhìn thấy chính xác 100% bối cảnh kỹ thuật mà không cần đoán mò.
- Yêu cầu sửa lỗi cục bộ (không xáo trộn kiến trúc toàn cục) và yêu cầu cung cấp test case kiểm thử biên.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Debug lỗi kích thước nhãn trong CrossEntropyLoss
Trong PyTorch, `nn.CrossEntropyLoss` yêu cầu nhãn mục tiêu phải là tensor 1D kiểu số nguyên (`torch.long`) có kích thước `[Batch]`. Nếu thí sinh truyền vào tensor 2D kiểu float `[Batch, 1]`, lỗi gì sẽ xảy ra? Hãy viết một gói Debug Prompt hoàn chỉnh để nhờ LLM giải thích lỗi này.

### Bài 2 (Easy): Tạo Synthetic Edge-Case kiểm thử hàm tiền xử lý
Viết một hàm `generate_edge_case_texts()` trả về danh sách các câu văn bản hiểm hóc:
- Chuỗi hoàn toàn rỗng `""`
- Chuỗi chỉ chứa toàn dấu cách `"   "`
- Chuỗi chỉ chứa toàn ký tự đặc biệt `@#$%^&*`
- Chuỗi có độ dài 10,000 từ
Chạy thử qua pipeline của bạn để xem hàm có bị sập (crash) không.

### Bài 3 (Medium): Phát hiện lỗi chia cho 0 trong chuẩn hóa dữ liệu
Khi một cột dữ liệu có tất cả các giá trị bằng nhau (độ lệch chuẩn $\sigma = 0$), công thức $x_{norm} = rac{x - \mu}{\sigma}$ sẽ gây ra lỗi `ZeroDivisionError` hoặc sinh ra các giá trị `NaN`/`inf`. Viết một prompt yêu cầu LLM đề xuất cách xử lý an toàn bằng epsilon số học ($\epsilon = 10^{-7}$).

### Bài 4 (Medium): Kiểm chứng chéo phương thức bằng `help()` và Docstrings
Khi LLM gợi ý một tham số mới trong PyTorch (ví dụ `weights_only=True` trong `torch.load`), hãy viết câu lệnh Python dùng `help(torch.load)` hoặc `torch.load.__doc__` để tự mình kiểm tra xem tham số đó có thực sự tồn tại trong phiên bản bạn đang cài hay không.

### Bài 5 (Challenge): Xây dựng Bộ kiểm tra Tự động Không có NaN (Zero-NaN Assurance Sandbox)
Viết một hàm bọc (Decorator) `@assert_no_nan` áp dụng cho bất kỳ hàm huấn luyện nào. Nếu sau khi chạy hàm mà trọng số mô hình hoặc giá trị mất mát xuất hiện bất kỳ phần tử `NaN` hay `Inf` nào, chương trình sẽ lập tức ngắt và in ra cảnh báo đỏ cùng trạng thái của batch dữ liệu hiện tại.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong áp lực phòng thi Olympic AI, khi bạn gặp một thông báo lỗi dài 40 dòng, điều tồi tệ nhất là ngồi đọc mò từng chữ một cách vô định. Một thí sinh có trang bị **Quy trình LLM Debugging Workflow** sẽ chỉ mất 15 giây: Copy Traceback -> Thu gọn thành MRE -> Đưa vào Prompt Template -> Nhận giải pháp sửa lỗi -> Kiểm thử trên Edge Case -> Tiếp tục thi đấu! Kỹ năng này có thể cứu vãn cả một kỳ thi của bạn.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh copy giải pháp sửa lỗi của LLM nhưng không đọc kỹ, khiến LLM âm thầm thay thế hàm mất mát `nn.CrossEntropyLoss` thành `nn.MSELoss` mà thí sinh không hề hay biết:

```python
# CODE CÓ LỖI DO CHẤP NHẬN SỬA ĐỔI BỪA BÃI CỦA LLM:
# De bai yeu cau: Phan loai da lop (can CrossEntropyLoss)
# LLM thay loi Shape nen tu y doi sang MSELoss de "khong bi bao loi shape nua"!
# Hau qua: Mo hinh chay khong bao loi nhung do chinh xac rot ve 0%!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao việc một đoạn code "chạy không báo lỗi" hoàn toàn khác với việc "mô hình học đúng bản chất toán học".
2. Khẳng định nguyên tắc: Tuyệt đối không cho phép LLM tự ý thay đổi hàm mất mát và mục tiêu của bài thi!



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Thuật ngữ MRE trong kỹ nghệ gỡ lỗi phần mềm viết tắt của cụm từ nào?
   - A. Maximum Resource Estimation
   - B. Minimal Reproducible Example (Mẫu Tái Lập Tối Thiểu)
   - C. Machine Recognition Engine
   - D. Model Regularization Error

2. Khi gửi thông báo lỗi cho LLM, hành động nào sau đây là TỐI KỴ và KÉM HIỆU QUẢ NHẤT?
   - A. Cung cấp toàn bộ Traceback chi tiết.
   - B. Chỉ ném mỗi câu: "Code bị lỗi, sửa đi!" mà không cung cấp thông báo lỗi, không cung cấp code tối thiểu và không nêu kích thước dữ liệu.
   - C. Cung cấp thông tin phiên bản hệ điều hành và thư viện.
   - D. Nêu giả thuyết cá nhân về nguyên nhân lỗi.

3. "Synthetic Edge Cases" (Các trường hợp kiểm thử biên nhân tạo) được thiết kế nhằm mục đích gì?
   - A. Làm cho máy tính chạy chậm lại.
   - B. Kiểm tra độ bền bỉ và khả năng chống chịu lỗi của mã nguồn trước các tình huống dữ liệu dị thường (như chuỗi rỗng, batch 1 mẫu, chia cho 0) trước khi nộp bài chính thức.
   - C. Tăng số lượng dòng code của notebook.
   - D. Thay thế cho việc thu thập dữ liệu thi đấu.

4. Phương thức nào trong Python giúp lập trình viên kiểm tra tài liệu và tham số chính thức của một hàm để đối soát xem gợi ý của LLM có phải là ảo giác hay không?
   - A. `print(math)`
   - B. `help(function_name)` hoặc `function_name.__doc__`
   - C. `delete(function_name)`
   - D. `exit()`

5. Khi LLM đề xuất một giải pháp sửa lỗi làm mất đi thông báo lỗi màu đỏ nhưng làm sai lệch bản chất toán học của bài toán (ví dụ đổi hàm mất mát), thái độ đúng đắn của thí sinh là gì?
   - A. Vui mừng nộp bài ngay lập tức.
   - B. Bác bỏ giải pháp đó, giữ vững hàm mục tiêu chuẩn mực và yêu cầu LLM tìm giải pháp sửa lỗi đúng đắn ở tầng kích thước dữ liệu (Tensor Reshaping).
   - C. Bỏ thi.
   - D. Tắt máy tính.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Quy trình 4 bước đóng gói báo cáo lỗi có cấu trúc (Structured Bug Report Packet).
  - Tầm quan trọng của Mẫu Tái Lập Tối Thiểu (Minimal Reproducible Example - MRE).
  - Cách dùng LLM sinh các bộ kiểm thử biên hiểm hóc (Synthetic Edge Cases).
  - Giao thức đối soát ảo giác API (Hallucination Cross-Verification).
* **What I can now do:**
  - Cô lập và giải quyết mọi lỗi sập code trong phòng thi một cách khoa học, bình tĩnh trong chưa đầy 60 giây.
  - Kiểm tra độ bền của pipeline trước khi bước vào giờ nộp bài chính thức.
* **What I still need to practice:**
  - Hoàn thiện sản phẩm cuối cùng của Module 6: **Personal Prompt Kit & Quy Trình Trợ Lực Thi Đấu** ở Buổi 3!



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Buổi tiếp theo (**M6_S03**) là buổi tổng kết đỉnh cao của Module 6: **Xây Dựng Personal Prompt Kit & Quy Trình Trợ Lực Thi Đấu Không Lệ Thuộc**! Bạn sẽ hoàn thành bộ vũ khí Prompt Kit cá nhân hóa cho toàn bộ 5 giai đoạn làm bài thi, hoàn thiện quy trình Debugging Workflow và nhận Chứng chỉ **Certificate of AI-assisted Problem Solving**!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M6**:
- Quy trình Debug code thi đấu có cấu trúc (Traceback + MRE) ✓
- Sinh Synthetic Edge-cases kiểm thử độ bền pipeline ✓
- Đối soát ảo giác API (Hallucination cross-verification protocol) ✓

* **Session Deliverable:** Mã nguồn Framework đóng gói lỗi và kiểm thử biên có trợ lực AI
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
